# 00_Configuracion

Configuración central del proyecto **accidentalidad-databricks**.

Este notebook define los parámetros compartidos del proyecto y crea el catálogo de Unity Catalog con los schemas de la arquitectura Medallion.

## Preparación reproducible
Crea catálogo, schemas Bronze/Silver/Gold/Quality y el Volume administrado `bronze.raw_files`.
Después cargue manualmente los cuatro CSV siguiendo `data/README.md`. Use un catálogo distinto
por entorno: el widget `entorno` es descriptivo y no proporciona aislamiento.
Se requieren privilegios para crear estos objetos en Unity Catalog.

## Dependencias compartidas con Shared Serverless
Abra el Git Folder completo en el editor y seleccione **Shared Serverless** en el selector
de compute de este notebook y de los demás notebooks del proyecto. La función requiere
que un administrador habilite **Project-Scoped Shared Compute** y un entorno versión 5 o
superior. Si no aparece Shared Serverless, serverless normal no puede compartir la instalación.

`pyproject.toml`, en la raíz del repositorio, declara las dependencias del entorno compartido.
La siguiente celda sincroniza ese entorno con `%uv sync`; después reinicia Python antes de
configurar Unity Catalog. Los demás notebooks del mismo Git Folder, usuario y compute
compartido utilizan las dependencias sin ejecutar comandos de instalación por notebook.
Las variables Python siguen siendo independientes en cada notebook.

Los Jobs creados mediante SDK necesitan configurar su propio entorno; esta preparación no
modifica automáticamente el entorno de esos Jobs ni de otro Git Folder.


In [ ]:
from pathlib import Path

project_root = next(
    (directory for directory in [Path.cwd(), *Path.cwd().parents]
     if (directory / "pyproject.toml").is_file()
     and (directory / "accidentalidad" / "policies.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Abra el Git Folder completo del proyecto: no se encontró pyproject.toml "
        "junto al paquete accidentalidad"
    )
print(f"Entorno compartido: {project_root / 'pyproject.toml'}")
print("Este notebook y los demás deben estar conectados a Shared Serverless")


In [ ]:
%uv sync


In [ ]:
# Recargar las dependencias del entorno compartido antes de configurar el catálogo.
dbutils.library.restartPython()


In [ ]:
from pathlib import Path
import sys

# Ejecutar desde el Git Folder completo, no desde un notebook importado aislado.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "accidentalidad" / "policies.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("No se encontró el paquete accidentalidad; abra el repositorio completo en un Git Folder")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from accidentalidad.policies import sql_identifier, parse_bool, nonnegative_int

dbutils.widgets.text("catalog_name", "accidentalidad_databricks", "Catálogo")
dbutils.widgets.dropdown("entorno", "training", ["training", "dev", "demo"], "Entorno")
PROY_CATALOG = sql_identifier(dbutils.widgets.get("catalog_name"))
PROY_ENTORNO = dbutils.widgets.get("entorno")
if PROY_ENTORNO not in {"training", "dev", "demo"}:
    raise ValueError("Entorno inválido")
spark.sql(f"CREATE CATALOG IF NOT EXISTS {PROY_CATALOG} COMMENT 'Accidentalidad Barranquilla'")
for schema in ["bronze", "silver", "gold", "quality"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {PROY_CATALOG}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {PROY_CATALOG}.bronze.raw_files")
RAW_PATH = f"/Volumes/{PROY_CATALOG}/bronze/raw_files"
print(f"Entorno: {PROY_ENTORNO}; cargar los cuatro CSV completos en {RAW_PATH}")

In [ ]:
# Verificación
display(spark.sql(f"SHOW SCHEMAS IN {PROY_CATALOG}"))